# Lesson 44 — SLAM: How a Robot Builds Its Own Map

The robot can already see, track, and measure distance. This notebook is the next question:

> **Where am I — and what does this place look like?**

SLAM means answering both at once. We will do the maths here, then overlay visual odometry on a free Full-HD clip you can post on LinkedIn.

Run the cells in order. For the finished 1920×1080 video, use the script at the bottom (it is faster than rendering inside Jupyter).

## 1. Four words people mix up

| Term | Question |
|---|---|
| **Mapping** | What does the environment look like? |
| **Localization** | Where am I on that map? |
| **Navigation** | Where do I need to go next? |
| **SLAM** | Build the map *and* estimate the pose, together |

GPS is not the answer indoors. Walls block satellites, and even outdoors GPS is metres-wrong — useless next to a table leg.

## 2. Odometry: integrate motion, every frame

A robot rarely gets its pose from a satellite. It asks: *how did I just move?* Wheel encoders, an IMU, or a camera can all answer that. Adding up those tiny steps is **odometry**.

In [ ]:
import math
from visual_slam import compose_pose, wrap_angle, loop_closure_triggered

x = y = theta = 0.0
for _ in range(10):
    x, y, theta = compose_pose(x, y, theta, forward=1.0, yaw=0.0)

print(f"Ten 1-metre steps straight ahead → ({x:.1f}, {y:.1f})")
print("That is odometry. It looks perfect — until the steps are slightly wrong.")

## 3. The problem: drift

Give every step a 2° heading error. After the same 10 metres the robot thinks it is still on the x-axis. It is not.

In [ ]:
x = y = theta = 0.0
for _ in range(10):
    x, y, theta = compose_pose(x, y, theta, forward=1.0, yaw=math.radians(2))

error = math.hypot(x - 10, y)
print(f"Believed pose: ({x:.2f}, {y:.2f}), heading {math.degrees(theta):.0f}°")
print(f"True pose if we had walked straight: (10.0, 0.0)")
print(f"Position error after 10 m: {error:.2f} m")
print("This error only ever grows. That is why a map built from odometry alone smears.")

## 4. Loop closure needs two things

Recognising the start of the run is not enough — you also have to have *left*. Otherwise the first few frames would all look like a loop.

In [ ]:
print("Near home, high match:     ", loop_closure_triggered(1.0, 0.90))
print("Far from home, low match:  ", loop_closure_triggered(12.0, 0.10))
print("Far from home, high match: ", loop_closure_triggered(12.0, 0.40))
print()
print("Only the last one is a loop closure: we have travelled AND we recognise the place.")

## 5. LiDAR SLAM vs visual SLAM vs visual-inertial

```text
LiDAR     →  laser ranges  →  occupancy grid     (robot vacuums, warehouses)
Camera    →  image features →  visual SLAM        (this notebook)
Camera+IMU → tightly coupled → visual-inertial    (phones, drones, ORB-SLAM3)
```

Same pipeline in every case:

```text
Sensors → odometry → matching → pose → map update → loop closure → optimised map
```

Names to recognise: **ORB-SLAM3**, **RTAB-Map**, **Cartographer**, **SLAM Toolbox** (the ROS 2 default for 2D lidar). We do not install them today.

## 6. Recap: Lesson 27's occupancy grid

Run this if you want the ASCII demo again — same room twice, once with a perfect pose and once with drifting odometry. The lidar never lies. The pose does.

In [ ]:
# Uncomment to reprint the Lesson 27 maps in this notebook:
# import slam_demo
# slam_demo.main()

## 7. Visual SLAM on a free HD clip

The default clip is Mixkit #21589 — a Full-HD walk down a library corridor. Commercial use is allowed. First run downloads ~55 MB into `video_out/` (gitignored).

This cell processes a **short slice** so the notebook stays interactive. The LinkedIn export at the bottom processes the whole clip.

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

from visual_slam import (
    STOCK_PATH,
    download_stock_video,
    seed_features,
    draw_tracks,
)

clip = download_stock_video()
cap = cv2.VideoCapture(str(clip))
ok, frame = cap.read()
assert ok, "Could not read the stock clip"

gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
pts = seed_features(gray)
print(f"Frame {frame.shape[1]}x{frame.shape[0]}  ·  {0 if pts is None else len(pts)} corners to track")

from collections import deque
tracks = [deque([(float(x), float(y))], maxlen=8) for x, y in pts.reshape(-1, 2)]
preview = draw_tracks(frame.copy(), tracks)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("What visual SLAM actually looks at — corners, not objects")
plt.show()
cap.release()

## 8. LinkedIn / reel export

The script adds a 2-second hook, the live dashboard (camera + growing map + pose), and a 3-second closer. Output is 1920×1080, which is what LinkedIn's native video player expects.

```bash
source .venv/bin/activate
python visual_slam.py --linkedin              # landscape mp4
python visual_slam.py --linkedin --reel --gif # + vertical + README gif
python visual_slam.py --video your.mp4 --linkedin
```

Files:

- `video_out/slam_linkedin_landscape.mp4` — post this on LinkedIn
- `video_out/slam_linkedin_landscape_vertical.mp4` — optional reel cut
- `assets/visual-slam-poster.jpg` — thumbnail

Caption and shot list: `docs/reel-scripts.md` (Reel 7).

In [ ]:
# Uncomment to render the full LinkedIn video from this notebook.
# Takes a minute or two on a MacBook Air M1.
#
# from visual_slam import run_visual_slam, download_stock_video, VIDEO_DIR
# run_visual_slam(
#     download_stock_video(),
#     VIDEO_DIR / "slam_linkedin_landscape.mp4",
#     make_reel=True,
#     make_gif=True,
# )

## Mini quiz

1. SLAM in full? → **Simultaneous Localization and Mapping**
2. LiDAR measures? → **Distance, by timing a laser**
3. Estimating the robot's pose? → **Localization**
4. Recognising a previous place and correcting the map? → **Loop closure**
5. Estimating motion from wheels or a camera? → **Odometry**

## Next — Lesson 45

Same ideas, a virtual LiDAR, inside Webots. Still no hardware.